In [1]:
import pickle
from pathlib import Path
totalseg_dir = Path("/nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/totalseg/")
with open(totalseg_dir / "totalseg_stats.pkl", "rb") as f:
    totalseg_stats = pickle.load(f) # case_id -> {label_id -> (bbox, volume, center)}

In [2]:
# Extract total occurrences and average volume for each label_id
from collections import defaultdict

label_stats = defaultdict(lambda: {"count": 0, "total_volume": 0.0})

for case_id, labels in totalseg_stats.items():
    for label_id, info in labels.items():
        label_stats[label_id]["count"] += 1
        label_stats[label_id]["total_volume"] += float(info["volume"])

# Calculate average volume and create summary
label_summary = {
    label_id: {
        "occurrences": stats["count"],
        "avg_volume": stats["total_volume"] / stats["count"]
    }
    for label_id, stats in label_stats.items()
}

# Display as a sorted DataFrame
import pandas as pd
df_label_stats = pd.DataFrame.from_dict(label_summary, orient="index")
df_label_stats.index.name = "label_id"
df_label_stats = df_label_stats.sort_values("occurrences", ascending=False)
df_label_stats

,occurrences,avg_volume
label_id,,
spinal_cord,1183,15882.996619
autochthon_left,1133,84760.713151
autochthon_right,1131,84166.481874
aorta,1104,55447.712862
esophagus,1033,8074.079380
...,...,...
vertebrae_C4,250,3465.568000
brain,249,204927.823293
vertebrae_C3,229,3623.135371


In [3]:
import sys
sys.path.append("/nfs/norasys/notebooks/camaret/ic_segmentation")
from data.label_ids_totalseg import category_map
category_map

{'esophagus': 'Organs (Abd/Pelvis)',
 'stomach': 'Organs (Abd/Pelvis)',
 'duodenum': 'Organs (Abd/Pelvis)',
 'small_bowel': 'Organs (Abd/Pelvis)',
 'colon': 'Organs (Abd/Pelvis)',
 'liver': 'Organs (Abd/Pelvis)',
 'gallbladder': 'Organs (Abd/Pelvis)',
 'pancreas': 'Organs (Abd/Pelvis)',
 'spleen': 'Organs (Abd/Pelvis)',
 'kidney_left': 'Organs (Abd/Pelvis)',
 'kidney_right': 'Organs (Abd/Pelvis)',
 'urinary_bladder': 'Organs (Abd/Pelvis)',
 'prostate': 'Organs (Abd/Pelvis)',
 'adrenal_gland_right': 'Organs (Abd/Pelvis)',
 'adrenal_gland_left': 'Organs (Abd/Pelvis)',
 'kidney_cyst_left': 'Organs (Abd/Pelvis)',
 'kidney_cyst_right': 'Organs (Abd/Pelvis)',
 'heart': 'Organs (Thorax/Head/Spine)',
 'lung_upper_lobe_left': 'Organs (Thorax/Head/Spine)',
 'lung_lower_lobe_left': 'Organs (Thorax/Head/Spine)',
 'lung_upper_lobe_right': 'Organs (Thorax/Head/Spine)',
 'lung_middle_lobe_right': 'Organs (Thorax/Head/Spine)',
 'lung_lower_lobe_right': 'Organs (Thorax/Head/Spine)',
 'trachea': 'Organs

In [ ]:
## add category names to the DataFrame
df_label_stats["category"] = df_label_stats.index.map(category_map)
df_label_stats = df_label_stats[["category", "occurrences", "avg_volume"]]

Index([], dtype='object', name='label_id')


Label split heuristic : paired-block

Rule: Always keep Left/Right pairs together (e.g., both Kidneys in Val, both Adrenals in Train).

Reason: If the model sees the left organ during training, segmentation of the right organ becomes a trivial symmetry task rather than a test of true generalization.

Regional Contiguity (No Serial Leakage):

Rule: For serial structures like Ribs and Vertebrae, split by anatomical region (e.g., Upper Ribs vs. Lower Ribs) rather than interleaving them (e.g., Rib 1 vs. Rib 2).

Reason: Adjacent structures share nearly identical local shapes. Splitting by region forces the model to generalize to different shapes (e.g., floating ribs vs. attached ribs) rather than just interpolating between neighbors.

Volume-Weighted Balancing:

Rule: Counter-weight massive organs against each other (e.g., Liver in Val vs. Lungs in Train) to equalize total occurrences and voxel volume.

Reason: This prevents one split from being dominated by "easy" large targets while the other is full of "hard" small targets, ensuring the difficulty is comparable.

In [12]:

split_dict = {
    "train": [
        "spinal_cord",  # Added
        "rib_left_1", "rib_right_1", "rib_left_2", "rib_right_2", 
        "rib_left_3", "rib_right_3", "rib_left_4", "rib_right_4", 
        "rib_left_5", "rib_right_5", "rib_left_6", "rib_right_6",
        "lung_upper_lobe_left", "lung_upper_lobe_right", 
        "lung_lower_lobe_left", "lung_lower_lobe_right", 
        "lung_middle_lobe_right", 
        "heart", "trachea", "thyroid_gland", 
        "stomach", "spleen", "pancreas", "gallbladder", 
        "adrenal_gland_left", "adrenal_gland_right", 
        "clavicula_left", "clavicula_right", 
        "scapula_left", "scapula_right", 
        "humerus_left", "humerus_right", 
        "hip_left", "hip_right", 
        "gluteus_maximus_left", "gluteus_maximus_right", 
        "gluteus_medius_left", "gluteus_medius_right", 
        "gluteus_minimus_left", "gluteus_minimus_right", 
        "aorta", "pulmonary_vein", 
        "subclavian_artery_left", "subclavian_artery_right", 
        "common_carotid_artery_left", "common_carotid_artery_right", 
        "vertebrae_C1", "vertebrae_C2", "vertebrae_C3", "vertebrae_C4", 
        "vertebrae_C5", "vertebrae_C6", "vertebrae_C7", 
        "vertebrae_T1", "vertebrae_T2", "vertebrae_T3", "vertebrae_T4", "vertebrae_T5"
    ],
    "val": [
        "vertebrae_S1", # Added
        "rib_left_7", "rib_right_7", "rib_left_8", "rib_right_8", 
        "rib_left_9", "rib_right_9", "rib_left_10", "rib_right_10", 
        "rib_left_11", "rib_right_11", "rib_left_12", "rib_right_12", 
        "sternum", "costal_cartilages", 
        "liver", 
        "kidney_left", "kidney_right", "kidney_cyst_left", "kidney_cyst_right", 
        "colon", "small_bowel", "duodenum", 
        "urinary_bladder", "prostate", "esophagus", 
        "brain", "atrial_appendage_left", 
        "femur_left", "femur_right", "skull", 
        "iliopsoas_left", "iliopsoas_right", 
        "autochthon_left", "autochthon_right", 
        "inferior_vena_cava", "superior_vena_cava", 
        "portal_vein_and_splenic_vein", 
        "iliac_artery_left", "iliac_artery_right", 
        "iliac_vena_left", "iliac_vena_right", 
        "brachiocephalic_vein_left", "brachiocephalic_vein_right", "brachiocephalic_trunk", 
        "vertebrae_T6", "vertebrae_T7", "vertebrae_T8", "vertebrae_T9", 
        "vertebrae_T10", "vertebrae_T11", "vertebrae_T12", 
        "vertebrae_L1", "vertebrae_L2", "vertebrae_L3", "vertebrae_L4", "vertebrae_L5", 
        "sacrum"
    ]
}
# add split info to the DataFrame
def get_split(label_id):
    for split, labels in split_dict.items():
        if label_id in labels:
            return split
    return "unknown"

df_label_stats["split"] = df_label_stats.index.map(get_split)
df_label_stats = df_label_stats[["category", "occurrences", "avg_volume", "split"]]
df_label_stats.to_csv(totalseg_dir / "label_stats.csv")

In [14]:
df_label_stats[df_label_stats["split"] == "train"].sort_values("occurrences", ascending=False)

,category,occurrences,avg_volume,split
label_id,,,,
spinal_cord,Organs (Thorax/Head/Spine),1183,15882.996619,train
aorta,Vessels,1104,55447.712862,train
lung_upper_lobe_left,Organs (Thorax/Head/Spine),1024,222578.120117,train
lung_lower_lobe_left,Organs (Thorax/Head/Spine),1006,198957.605368,train
lung_lower_lobe_right,Organs (Thorax/Head/Spine),991,228107.647830,train
heart,Organs (Thorax/Head/Spine),924,147998.127706,train
rib_left_6,Bones (Ribs/Sternum),909,4891.446645,train
stomach,Organs (Abd/Pelvis),905,84356.187845,train
lung_upper_lobe_right,Organs (Thorax/Head/Spine),894,209568.449664,train


In [15]:
df_label_stats[df_label_stats["split"] == "val"].sort_values("occurrences", ascending=False)

,category,occurrences,avg_volume,split
label_id,,,,
autochthon_left,Muscles,1133,84760.713151,val
autochthon_right,Muscles,1131,84166.481874,val
esophagus,Organs (Abd/Pelvis),1033,8074.079380,val
costal_cartilages,Bones (Ribs/Sternum),1017,33461.928220,val
inferior_vena_cava,Vessels,977,17155.522006,val
sternum,Bones (Ribs/Sternum),959,14121.300313,val
liver,Organs (Abd/Pelvis),939,392488.880724,val
colon,Organs (Abd/Pelvis),913,155435.303395,val
rib_left_7,Bones (Ribs/Sternum),891,5314.369248,val
